In [4]:
from mpi4py import MPI
import numpy as np
import ufl
from dolfinx.mesh import create_unit_square
from dolfinx.fem import functionspace, Function, Constant, dirichletbc, locate_dofs_geometrical
from dolfinx.fem.petsc import LinearProblem, NonlinearProblem
from basix.ufl import element
from dolfinx.io import VTXWriter

# ----------------------------
# 1. Create mesh
# ----------------------------
n_elem = 32
msh = create_unit_square(MPI.COMM_WORLD, n_elem, n_elem)
tdim = msh.topology.dim
gdim = msh.geometry.dim

# Store original coordinates
x0 = msh.geometry.x.copy()

# ----------------------------
# 2. Scalar function space for solution
# ----------------------------
V = functionspace(msh, ("Lagrange", 1))
u, v = Function(V), ufl.TestFunction(V)
u_n = Function(V)

# Poisson source
f = Constant(msh, 1.0)
dt = Constant(msh, 0.1)
F = (u - u_n)/dt * v * ufl.dx + ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx
F += -f * v * ufl.dx

# ----------------------------
# 3. Dirichlet BC on left/bottom edges
# ----------------------------
dofs_D = locate_dofs_geometrical(
    V, lambda x: np.logical_or(np.isclose(x[0], 0), np.isclose(x[1], 0))
)
u_bc = Function(V)
u_bc.interpolate(lambda x: np.zeros(x.shape[1]))
bcs = [dirichletbc(u_bc, dofs_D)]

# ----------------------------
# 4. Vector function space for mesh motion (ALE)
# ----------------------------
V_vec = functionspace(
    msh,
    element("Lagrange", msh.topology.cell_name(), 1, shape=(gdim,))
)
u_mesh = Function(V_vec)

problem = NonlinearProblem(F, u, bcs=bcs, petsc_options_prefix="demo_poisson_",
    petsc_options={"ksp_type": "preonly", "pc_type": "lu", "ksp_error_if_not_converged": True},
)

# ----------------------------
# 5. Time loop with moving mesh and VTU export
# ----------------------------

writer = VTXWriter(MPI.COMM_WORLD, "results/moving_mesh.bp", u, mesh_policy=0)
# Optionally write initial mesh
# vtk.write_mesh(msh)
t = 0.0
while t < 20:
    t += float(dt)
    # --- 5a. Define mesh motion ---
    def mesh_motion(x):
        # x.shape = (gdim, num_points)
        values = np.zeros((gdim, x.shape[1]))
        values[1] = 0.1 * np.sin(np.pi * (x[0] - t/2))  # vertical displacement
        return values

    u_mesh.interpolate(mesh_motion)

    # --- 5b. Update mesh coordinates ---
    array_reshaped = u_mesh.x.array.reshape(msh.geometry.x.shape[0], gdim)
    zero_col = np.zeros((array_reshaped.shape[0], 1))
    array_appended = np.hstack([array_reshaped, zero_col])

    msh.geometry.x[:] = x0 + array_appended

    # --- 5d. Solve Poisson problem ---

    problem.solve()

    u_n.x.array[:] = u.x.array[:]

    # --- 5e. Write solution at current timestep ---
    writer.write(t)
    print(t)

0.1
0.2
0.30000000000000004
0.4
0.5
0.6
0.7
0.7999999999999999
0.8999999999999999
0.9999999999999999
1.0999999999999999
1.2
1.3
1.4000000000000001
1.5000000000000002
1.6000000000000003
1.7000000000000004
1.8000000000000005
1.9000000000000006
2.0000000000000004
2.1000000000000005
2.2000000000000006
2.3000000000000007
2.400000000000001
2.500000000000001
2.600000000000001
2.700000000000001
2.800000000000001
2.9000000000000012
3.0000000000000013
3.1000000000000014
3.2000000000000015
3.3000000000000016
3.4000000000000017
3.5000000000000018
3.600000000000002
3.700000000000002
3.800000000000002
3.900000000000002
4.000000000000002
4.100000000000001
4.200000000000001
4.300000000000001
4.4
4.5
4.6
4.699999999999999
4.799999999999999
4.899999999999999
4.999999999999998
5.099999999999998
5.1999999999999975
5.299999999999997
5.399999999999997
5.4999999999999964
5.599999999999996
5.699999999999996
5.799999999999995
5.899999999999995
5.999999999999995
6.099999999999994
6.199999999999994
6.29999999999